## move data from drive to colab


In [ ]:
cd /content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026

/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026


In [ ]:
cp -r '/content/drive/MyDrive/cv_22641171_NgoTruongDinh/lab_bo_sung_10 03 2026/new_Data/Covid19-Pneumonia-Normal Chest X-Ray Images Dataset.zip' /content

# benchmark

In [ ]:
from functools import lru_cache

import cv2
import numpy as np

def logistic_map_permutation(n, seed, r=3.99):
    """Generate a block permutation using a logistic chaotic map."""
    x = (seed % 1000) / 1000.0

    if x in (0, 0.25, 0.5, 0.75, 1):
        x += 0.123

    for _ in range(100):
        x = r * x * (1.0 - x)

    sequence = np.empty(n, dtype=np.float64)

    for i in range(n):
        x = r * x * (1.0 - x)
        sequence[i] = x

    return np.argsort(sequence)


@lru_cache(maxsize=None)
def _cached_dct_perturbation(
    height,
    width,
    seed,
    flip_prob=0.10,
    noise_scale=1.0
):
    """
    Cache flips and Gaussian noise.

    The generated values are identical to the original implementation
    because the same seed and the same random-call order are retained.
    """
    hf_height = height - height // 2
    hf_width = width - width // 2
    shape = (hf_height, hf_width)

    rng = np.random.default_rng(seed)

    flips = rng.choice(
        [-1.0, 1.0],
        size=shape,
        p=[flip_prob, 1.0 - flip_prob]
    ).astype(np.float32)

    noise = rng.normal(
        0.0,
        noise_scale,
        size=shape
    ).astype(np.float32)

    return flips, noise


def dct_process_block(
    block,
    seed,
    flip_prob=0.10,
    noise_scale=1.0
):
    block_f = np.asarray(block, dtype=np.float32)
    coefficients = cv2.dct(block_f)

    height, width = coefficients.shape
    hf = (
        slice(height // 2, height),
        slice(width // 2, width)
    )

    flips, noise = _cached_dct_perturbation(
        height,
        width,
        seed,
        flip_prob,
        noise_scale
    )

    coefficients[hf] *= flips
    coefficients[hf] += noise

    return cv2.idct(coefficients)


def chaos_spectral_scramble(img, block_size, seed):
    """
    Chaos-based block permutation followed by channel-wise DCT processing.
    """
    height, width, channels = img.shape

    if height % block_size != 0 or width % block_size != 0:
        raise ValueError(
            f"Image dimensions ({height}, {width}) must be divisible "
            f"by block_size={block_size}."
        )

    height_blocks = height // block_size
    width_blocks = width // block_size
    num_blocks = height_blocks * width_blocks

    # Shape:
    # HWC -> (height_blocks, width_blocks, block_size, block_size, C)
    blocks = (
        img.reshape(
            height_blocks,
            block_size,
            width_blocks,
            block_size,
            channels
        )
        .transpose(0, 2, 1, 3, 4)
        .reshape(
            num_blocks,
            block_size,
            block_size,
            channels
        )
    )

    permutation = logistic_map_permutation(num_blocks, seed)
    shuffled_blocks = blocks[permutation]

    processed_blocks = np.empty(
        shuffled_blocks.shape,
        dtype=np.float32
    )

    for block_index in range(num_blocks):
        for channel in range(channels):
            processed_blocks[
                block_index, :, :, channel
            ] = dct_process_block(
                shuffled_blocks[
                    block_index, :, :, channel
                ],
                seed + block_index + channel
            )

    # Convert blocks back to HWC image layout
    output = (
        processed_blocks.reshape(
            height_blocks,
            width_blocks,
            block_size,
            block_size,
            channels
        )
        .transpose(0, 2, 1, 3, 4)
        .reshape(height, width, channels)
    )

    return np.clip(output, 0, 255).astype(np.uint8)


def msb_lsb_only(img, k=3):
    img = np.asarray(img, dtype=np.uint8)

    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError("Input image must have shape (H, W, 3).")

    msb_mask = ((1 << k) - 1) << (8 - k)
    lsb_mask = (1 << (8 - k)) - 1

    msb = (img & msb_mask) >> (8 - k)
    lsb = img & lsb_mask

    # Process all three channels at once.
    mixed_msb = msb.copy()

    mixed_msb[:-1, :, :] ^= msb[1:, :, :]
    mixed_msb[:, :-1, :] ^= msb[:, 1:, :]

    # Channel mapping: 0->1, 1->2, 2->0
    mixed_msb ^= np.roll(msb, shift=-1, axis=2)

    encrypted = (
        mixed_msb << (8 - k)
    ) | lsb

    return encrypted.astype(np.uint8)


def apply_checkerboard_lsb_inversion(
    img,
    block_size,
    msb_mask,
    lsb_mask
):
    """
    Vectorized replacement for the two nested block-XOR loops.
    """
    height, width, channels = img.shape

    if height % block_size != 0 or width % block_size != 0:
        raise ValueError(
            f"Image dimensions ({height}, {width}) must be divisible "
            f"by block_size={block_size}."
        )

    height_blocks = height // block_size
    width_blocks = width // block_size

    blocks = (
        img.reshape(
            height_blocks,
            block_size,
            width_blocks,
            block_size,
            channels
        )
        .transpose(0, 2, 1, 3, 4)
    )

    row_indices = np.arange(height_blocks)[:, None]
    column_indices = np.arange(width_blocks)[None, :]
    checkerboard_mask = (
        (row_indices + column_indices) % 2 == 1
    )

    selected = blocks[checkerboard_mask]

    lsb_part = selected & lsb_mask
    msb_part = selected & msb_mask

    blocks[checkerboard_mask] = (
        msb_part | (lsb_mask - lsb_part)
    )

    return (
        blocks.transpose(0, 2, 1, 3, 4)
        .reshape(height, width, channels)
    )


def msb_lsb_block_xor(img, k=3, block_xor=4):
    img = np.asarray(img, dtype=np.uint8)

    msb_mask = ((1 << k) - 1) << (8 - k)
    lsb_mask = (1 << (8 - k)) - 1

    encrypted = msb_lsb_only(img, k=k)

    encrypted = apply_checkerboard_lsb_inversion(
        encrypted,
        block_size=block_xor,
        msb_mask=msb_mask,
        lsb_mask=lsb_mask
    )

    return encrypted.astype(np.uint8)


def advanced_learnable_encrypt(
    img,
    k=3,
    block_xor=4,
    block_shuffle=16,
    seed=2024
):
    encrypted = msb_lsb_block_xor(
        img,
        k=k,
        block_xor=block_xor
    )

    return chaos_spectral_scramble(
        encrypted,
        block_size=block_shuffle,
        seed=seed
    )

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

# import encryption


SRC_ROOT = "/content/data_processed"

IMG_SIZE = 224
K = 3
BLOCK_XOR = 4
BLOCK_SHUFFLE = 16
SEED = 2024

NUM_WARMUP = 10
MAX_IMAGES = None  # ví dụ 500; None = toàn bộ dataset


def load_image(path):
    with Image.open(path) as im:
        im = im.convert("RGB")

        if im.size != (IMG_SIZE, IMG_SIZE):
            im = im.resize(
                (IMG_SIZE, IMG_SIZE),
                resample=Image.BILINEAR
            )

        return np.asarray(im, dtype=np.uint8)


def benchmark_encryption_only(
    src_root,
    max_images=None,
    num_warmup=10
):
    dataset = ImageFolder(src_root)
    image_paths = [path for path, _ in dataset.samples]

    if max_images is not None:
        image_paths = image_paths[:max_images]

    print(f"Number of benchmark images: {len(image_paths)}")

    # Load trước toàn bộ ảnh để loại bỏ thời gian I/O
    images = [
        load_image(path)
        for path in tqdm(image_paths, desc="Loading images")
    ]

    if not images:
        raise ValueError("Không tìm thấy ảnh để benchmark.")

    # Warm-up để tránh lần chạy đầu tiên làm sai lệch kết quả
    warmup_count = min(num_warmup, len(images))

    for image in images[:warmup_count]:
        _ = advanced_learnable_encrypt(
            image,
            k=K,
            block_xor=BLOCK_XOR,
            block_shuffle=BLOCK_SHUFFLE,
            seed=SEED
        )

    times_ms = []

    for image in tqdm(images, desc="Benchmarking encryption"):
        start = time.perf_counter()

        encrypted_image = advanced_learnable_encrypt(
            image,
            k=K,
            block_xor=BLOCK_XOR,
            block_shuffle=BLOCK_SHUFFLE,
            seed=SEED
        )

        elapsed = time.perf_counter() - start
        times_ms.append(elapsed * 1000)

        # Đảm bảo kết quả thực sự được tạo
        if encrypted_image is None:
            raise RuntimeError("Encryption returned None.")

    times_ms = np.asarray(times_ms, dtype=np.float64)

    results = {
        "num_images": len(times_ms),
        "mean_ms_per_image": times_ms.mean(),
        "std_ms_per_image": times_ms.std(ddof=1)
        if len(times_ms) > 1 else 0.0,
        "median_ms_per_image": np.median(times_ms),
        "min_ms_per_image": times_ms.min(),
        "max_ms_per_image": times_ms.max(),
        "throughput_images_per_second": 1000.0 / times_ms.mean(),
        "total_encryption_time_seconds": times_ms.sum() / 1000.0
    }

    print("\n=== Encryption-only benchmark ===")

    for key, value in results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

    pd.DataFrame({
        "image_index": np.arange(1, len(times_ms) + 1),
        "encryption_time_ms": times_ms
    }).to_csv(
        "encryption_time_per_image.csv",
        index=False
    )

    pd.DataFrame([results]).to_csv(
        "encryption_benchmark_summary.csv",
        index=False
    )

    return results, times_ms


results, times = benchmark_encryption_only(
    SRC_ROOT,
    max_images=MAX_IMAGES,
    num_warmup=NUM_WARMUP
)

Number of benchmark images: 5228


Loading images:   0%|          | 0/5228 [00:00<?, ?it/s]

Benchmarking encryption:   0%|          | 0/5228 [00:00<?, ?it/s]


=== Encryption-only benchmark ===
num_images: 5228
mean_ms_per_image: 18.4343
std_ms_per_image: 0.5286
median_ms_per_image: 18.2809
min_ms_per_image: 17.8151
max_ms_per_image: 25.3622
throughput_images_per_second: 54.2467
total_encryption_time_seconds: 96.3746
